In [1]:
# Importing things I need
import scirpy as ir
import scanpy as sc
from glob import glob
import pandas as pd
import tarfile
import anndata
import warnings
import scanpy as sc
import anndata as an
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Figure 3E: TCR overlap across activated CD8 T cell clusters

In [2]:
# Importing CD8 dataframes and TCR dataframes
cd8_pre_label_transfer = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/HC_Tclus/cd8t_quick_cluster2.csv', sep=',')
cd8_post_label_transfer = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/HC_Tclus/cd8_post_label_transfer.csv', sep = ',')
filtered_all_tcr = pd.read_csv("gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/meta_patient_TCRs/filtered_all_tcr_042025.csv", sep = ',')

mhc2_cd8_pre_lt = cd8_pre_label_transfer[cd8_pre_label_transfer['sub_humap_fgraph_res.1'] == 4]
cxcl13_cd8_pre_lt = cd8_pre_label_transfer[cd8_pre_label_transfer['sub_humap_fgraph_res.1'] == 7]
hobit_cd8_pre_lt = cd8_pre_label_transfer[cd8_pre_label_transfer['sub_humap_fgraph_res.1'] == 8]

mhc2_cd8_pre_lt.to_csv("mhc2_cd8_pre_lt.csv", index=True) 
cxcl13_cd8_pre_lt.to_csv("cxcl13_cd8_pre_lt.csv", index=True) 
hobit_cd8_pre_lt.to_csv("hobit_cd8_pre_lt.csv", index=True) 

all_cells = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/data_minGenes_noDnsmpl/metadata_minGenes_noDnsmpl.txt.gz', sep='\t')
# act_cd8_post_lt = cd8_post_label_transfer[cd8_post_label_transfer['transferCd8tCellType'].isin([4,7,8])]
# act_cd8_post_lt.to_csv("act_cd8_post_lt.csv", index=True) 

# cd4_pre_label_transfer = pd.read_csv('cd4_t_full.csv', sep=',')



/tmp/ipykernel_276/3742148665.py:4: DtypeWarning: Columns (23,24,25,26,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  filtered_all_tcr = pd.read_csv("gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/meta_patient_TCRs/filtered_all_tcr_042025.csv", sep = ',')


In [3]:
# the barcode/cell_id is identical from scRNAseq and scTCRseq: however, after the dash  differs. 
# From here on out, TCRs and cells are matched based on barcode/cell_id AND the specimenID.
def splice_id_v1(df):
    df['cell_id'] = df['cell_id'].str.split('-').str[0]
    return df
cd8_pre_lt_meta = (splice_id_v1(cd8_pre_label_transfer))
cd8_post_lt_meta = (splice_id_v1(cd8_post_label_transfer))

filtered_all_tcr['barcode'] = filtered_all_tcr['barcode'].str.split('-').str[0]

In [4]:
# merge cd8 metadata with tcr metadata after the splicing
matching_cells_pre_lt = pd.merge(
    cd8_pre_lt_meta,
    filtered_all_tcr,
    left_on=['cell_id', 'specimenID'],
    right_on=['barcode', 'patient 10x channel'], 
    how='inner'
)

matching_cells_post_lt = pd.merge(
    cd8_post_lt_meta,
    filtered_all_tcr,
    left_on=['cell_id', 'specimenID'],  
    right_on=['barcode', 'patient 10x channel'],  
    how='inner'
)

In [10]:
# subsetting 
cd8_w_tcr = matching_cells_pre_lt
other_cd8_w_tcr = cd8_w_tcr[cd8_w_tcr['sub_humap_fgraph_res.1'] != 8] # or 4 or 7 which are the MHCII and CXCL13 clusters
print(len(other_cd8_w_tcr))
print(len(cd8_w_tcr))


# switch between pre/post to get proportions of specific clusters of cells under pre/post 
# PRE label transfer conditions for cells with TCR alpha Beta sequence

filter_values = [4,7,8] # we're interested in the activated T cell clusters [4,7,8] 
activated_cells_pre_lt = matching_cells_pre_lt[matching_cells_pre_lt['sub_humap_fgraph_res.1'].isin(filter_values)]

act_pre_lt_pre = activated_cells_pre_lt[activated_cells_pre_lt['treatment'] == 'Pre']
act_pre_lt_on = activated_cells_pre_lt[activated_cells_pre_lt['treatment'] == 'On']

act_cd8_pre_lt_g6 = activated_cells_pre_lt[activated_cells_pre_lt['PFSmo'] > 6]
act_cd8_pre_lt_l6 = activated_cells_pre_lt[activated_cells_pre_lt['PFSmo'] <= 6]

act_g6_pre_pre_lt =  act_pre_lt_pre[act_pre_lt_pre['PFSmo'] > 6]
act_g6_on_pre_lt = act_pre_lt_on[act_pre_lt_on['PFSmo'] > 6]
act_l6_pre_pre_lt = act_pre_lt_pre[act_pre_lt_pre['PFSmo'] <= 6]
act_l6_on_pre_lt = act_pre_lt_on[act_pre_lt_on['PFSmo'] <= 6]

act_cd8_w_tcr = activated_cells_pre_lt
mhcii_cd8_tcr = act_cd8_w_tcr[act_cd8_w_tcr['sub_humap_fgraph_res.1'] == 4]
cxcl13_cd8_tcr = act_cd8_w_tcr[act_cd8_w_tcr['sub_humap_fgraph_res.1'] == 7]
hobit_cd8_tcr = act_cd8_w_tcr[act_cd8_w_tcr['sub_humap_fgraph_res.1'] == 8]
print(len(mhcii_cd8_tcr))
print(len(cxcl13_cd8_tcr))
print(len(hobit_cd8_tcr))

25211
26361
3205
1923
1150


In [7]:
# Testing whether TCR-matched cells are enriched in a specific CD8 T-cell cluster relative to what would be 
# expected by chance alone. Here, we focused on the activated clusters (MHCII (ISGs) CD8, Tex (CXCL13+) CD8, 
# ZNF683+ CD8)

# First, `calculate_cluster_proportion` first identifies cells whose (specimenID, cdr3_beta) 
# clonotype pairs overlap with a reference set of CD8 T-cell clonotypes (`matching_cd8_cluster`). 
# Among these matched cells, it calculates the proportion that fall within a specified cluster 
# (e.g., cluster 13 defined by `sub_humap_fgraph_res.1`), would loop through all cluster numbers.

# Subsequently, `generate_null_distribution_overlap` constructs a permutation-based null model to assess
# whether the observed enrichment is greater than random expectation. In each iteration (1000 total), the
# `cdr3_beta` clonotype labels are randomly permuted across cells in `matching_cells`, which
# preserves the total number of cells and cluster assignments but breaks the true relationship
# between clonotype identity and cluster membership. The cluster proportion is recalculated
# after each permutation.

def calculate_cluster_proportion(matching_cells, matching_cd8_cluster, cluster_column='sub_humap_fgraph_res.1', target_cluster= 13):
    subset_matching_TCR = matching_cells[
        matching_cells.set_index(['specimenID', 'cdr3_beta']).index.isin(
            matching_cd8_cluster.set_index(['specimenID', 'cdr3_beta']).index
        )
    ]
    
    total_matching_cells = len(subset_matching_TCR)
    cluster_counts = subset_matching_TCR[cluster_column].value_counts()
    count_in_target_cluster = cluster_counts.get(target_cluster, 0)
    
    proportion_in_target_cluster = count_in_target_cluster / total_matching_cells if total_matching_cells > 0 else 0
    
    return proportion_in_target_cluster
def generate_null_distribution_overlap(matching_cells, matching_cd8_cluster, cluster_column='sub_humap_fgraph_res.1', target_cluster = 13, num_iterations=1000):
    null_distribution = []
    scrambled_data = matching_cells.copy()

    for i in range(num_iterations):
        scrambled_data['cdr3_beta'] = np.random.permutation(scrambled_data['cdr3_beta'])
        proportion_in_target_cluster = calculate_cluster_proportion(scrambled_data, matching_cd8_cluster, cluster_column, target_cluster)
        print(proportion_in_target_cluster)
        null_distribution.append(proportion_in_target_cluster)

    return null_distribution

In [11]:
# Running the above functions and computing Z-scores (can change hobit_cd8_tcr to mhcii_cd8_tcr, cxcl13_cd8_tcr)
from scipy.stats import norm
obs_value = calculate_cluster_proportion(other_cd8_w_tcr, hobit_cd8_tcr)
null_distribution = generate_null_distribution_overlap(other_cd8_w_tcr, hobit_cd8_tcr)

null_mean = np.mean(null_distribution)
null_std = np.std(null_distribution)

z_score = (obs_value - null_mean) / null_std if null_std > 0 else np.nan

# Compute two-tailed p-value
p_value = 2 * norm.sf(abs(z_score))  # Two-tailed test

0.00772626931567329
0.006772009029345372
0.006622516556291391
0.007641921397379912
0.013856812933025405
0.007717750826901874
0.005411255411255411
0.0034883720930232558
0.0055617352614015575
0.007658643326039387
0.007376185458377239
0.00558659217877095
0.010857763300760043
0.004250797024442083
0.007882882882882882
0.00547645125958379
0.006600660066006601
0.006329113924050633
0.005512679162072767
0.003157894736842105
0.007717750826901874
0.008958566629339306
0.004662004662004662
0.006396588486140725
0.006430868167202572
0.009814612868047983
0.005649717514124294
0.006607929515418502
0.010672358591248666
0.0056433408577878106
0.006872852233676976
0.010460251046025104
0.010964912280701754
0.007847533632286996
0.0031779661016949155
0.0032119914346895075
0.00331858407079646
0.00558659217877095
0.004561003420752566
0.0055617352614015575
0.004424778761061947
0.00562429696287964
0.005296610169491525
0.007927519818799546
0.006622516556291391
0.00589622641509434
0.0043859649122807015
0.00782122905

In [13]:
# can iterate through all combinations as specificied above to get raw p_value, subsequently adjusted by mulitplying
# by the number of CD8 clusters total (14) 
print(obs_value)
print(null_mean)
print(null_std)
print(z_score)
print(p_value)

0.02385231171377226
0.005908841010345884
0.002476616764032188
7.245154342819093
4.319470306447812e-13


# CD8 T cell gene signature scores: Figure 3G (part of 3H and 3I), Figure S3E

In [14]:
# reimporting again
cd8_pre_lt = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/HC_Tclus/cd8t_quick_cluster2.csv', sep=',')

In [15]:
# merging cd8 dataframe with adata that has been filtered to just include all T cells 
cluster_col = "sub_humap_fgraph_res.1"

def merge_cell_df_into_adata_obs(
    adata,
    cell_df,
    cols=None,
    key_df=None,          
    key_adata=None,  
    overwrite=False,
    verbose=True,):
    df = cell_df.copy()

    # choose columns to merge
    if cols is None:
        cols = list(df.columns)
    else:
        cols = [c for c in cols if c in df.columns]
        if len(cols) == 0:
            raise ValueError("None of the requested `cols` exist in cell_df.")

    # Case 1: merge by index (cell_df.index -> adata.obs_names)
    can_index_join = df.index.is_unique and pd.Index(df.index).isin(adata.obs_names).mean() > 0.50
    if key_df is None and key_adata is None and can_index_join:
        df_sub = df[cols].copy()
        # align exactly to adata.obs_names
        df_sub = df_sub.reindex(adata.obs_names)

        # avoid clobbering
        for c in cols:
            if (c in adata.obs.columns) and (not overwrite):
                if verbose:
                    print(f"[merge] skipping existing column (overwrite=False): {c}")
                df_sub = df_sub.drop(columns=[c])

        adata.obs = adata.obs.join(df_sub, how="left")

        meta = {
            "mode": "index_join",
            "n_cells_adata": int(adata.n_obs),
            "n_cells_df": int(df.shape[0]),
            "frac_df_index_in_adata": float(pd.Index(df.index).isin(adata.obs_names).mean()),
            "n_cols_added": int(df_sub.shape[1]),
        }
        if verbose:
            print("[merge] merged by index into adata.obs")
            print(meta)
        return adata, meta

    # Case 2: merge by explicit keys
    if key_df is None or key_adata is None:
        raise ValueError(
            "Could not confidently merge by index. "
            "Provide `key_df` (column in cell_df) and `key_adata` (column in adata.obs) "
            "that both contain the same cell IDs."
        )

    if key_adata not in adata.obs.columns:
        raise ValueError(f"key_adata='{key_adata}' not found in adata.obs columns.")
    if key_df not in df.columns:
        raise ValueError(f"key_df='{key_df}' not found in cell_df columns.")

    df_sub = df[[key_df] + cols].copy()
    df_sub = df_sub.drop_duplicates(subset=[key_df])

    # avoid clobbering
    cols_to_add = []
    for c in cols:
        if (c in adata.obs.columns) and (not overwrite):
            if verbose:
                print(f"[merge] skipping existing column (overwrite=False): {c}")
        else:
            cols_to_add.append(c)

    df_sub = df_sub[[key_df] + cols_to_add]

    merged = adata.obs.reset_index().rename(columns={"index": "__obsname__"})
    merged = merged.merge(df_sub, how="left", left_on=key_adata, right_on=key_df)
    merged = merged.set_index("__obsname__")
    merged = merged.drop(columns=[key_df])

    adata.obs = merged

    frac_matched = float(adata.obs[cols_to_add].notna().mean().mean()) if cols_to_add else 1.0
    meta = {
        "mode": "key_merge",
        "key_adata": key_adata,
        "key_df": key_df,
        "n_cells_adata": int(adata.n_obs),
        "n_cells_df": int(df.shape[0]),
        "n_cols_added": int(len(cols_to_add)),
        "avg_nonnull_rate_added_cols": frac_matched,
    }
    if verbose:
        print("[merge] merged by keys into adata.obs")
        print(meta)
    return adata, meta

adata_T = sc.read_h5ad("adata_T.h5ad")
adata_CD8_T, merge_meta = merge_cell_df_into_adata_obs(
    adata=adata_T,
    cell_df=cd8_pre_lt,
    cols=[cluster_col],
    key_df="cell_id",        
    key_adata="Unnamed: 0",    
    overwrite=True,
    verbose=True,
)


[merge] merged by keys into adata.obs
{'mode': 'key_merge', 'key_adata': 'Unnamed: 0', 'key_df': 'cell_id', 'n_cells_adata': 134270, 'n_cells_df': 43711, 'n_cols_added': 1, 'avg_nonnull_rate_added_cols': 0.3255455425634915}


In [16]:
# subset adata to only the cells present in cd8_pre_lt
cells_keep = pd.Index(cd8_pre_lt["cell_id"].astype(str).unique())

adata_keys = adata_CD8_T.obs["Unnamed: 0"].astype(str)

mask = adata_keys.isin(cells_keep).values
adata_CD8_T = adata_CD8_T[mask].copy()

print("After subsetting:", adata_CD8_T.n_obs, "cells")
print("Expected (unique df cells):", len(cells_keep))


After subsetting: 43711 cells
Expected (unique df cells): 43711


In [20]:
# Compute gene signature score activity per cell using Scanpy gene-set scoring, then pseudobulk scores to the specimen 
# level (mean/median across cells). Paired D01 vs D15 changes are calculated per patient, and significance is 
# tested using a paired Wilcoxon test overall and by PFS > 6 months or < 6 months group with BH-FDR correction.
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def _get_X_and_genes(adata, use_raw=False, layer=None):
    if use_raw and getattr(adata, "raw", None) is not None:
        X = adata.raw.X
        var_names = adata.raw.var_names
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names
    return X, var_names


def _score_genes_per_cell_scanpy(
    adata,
    genes,
    score_key="MAPK_score",
    use_raw=False,
    layer=None,
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Uses scanpy.sc.tl.score_genes to compute a per-cell gene-set score.
    Returns (scores_1d, genes_present, adata_scored).
    """
    ad = adata.copy()

    # If a layer is requested, score on that layer by temporarily swapping X.
    # (scanpy's score_genes historically scores on .X; use_raw overrides.)
    if layer is not None:
        if use_raw:
            raise ValueError("If use_raw=True, please do not pass layer (score_genes will use .raw).")
        if layer not in ad.layers:
            raise ValueError(f"layer='{layer}' not found in adata.layers")
        ad.X = ad.layers[layer]

    # Determine which gene names we are matching against
    if use_raw and getattr(ad, "raw", None) is not None:
        var_names = np.asarray(ad.raw.var_names)
        use_raw_arg = True
    else:
        var_names = np.asarray(ad.var_names)
        use_raw_arg = False

    gene_set = set(var_names.tolist())
    genes_present = [g for g in genes if g in gene_set]
    if len(genes_present) == 0:
        raise ValueError("None of the provided genes were found in adata.var_names (or adata.raw.var_names).")

    # Score genes
    sc.tl.score_genes(
        ad,
        gene_list=genes_present,
        score_name=score_key,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
        use_raw=use_raw_arg,
        copy=False,
    )

    scores = ad.obs[score_key].to_numpy(dtype=float)
    return scores, genes_present, ad

def mapk_change_stats_from_adata(
    adata_T,
    genes,
    specimen_col="specimenID",
    cluster_col="L2_humap_fgraph_res.0.4",
    exclude_clusters=None,
    patient_parser=lambda s: s.str.slice(0, 3),
    day_parser=lambda s: s.str.slice(3, 6),
    d01="D01",
    d15="D15",
    nonresponder_patients=None,
    responder_patients=None,
    exclude_patients=None,
    use_raw=False,
    layer=None,
    pseudobulk_agg="median",
    dropna_pairs=True,
    run_overall=True,
    run_by_group=True,
    fdr_method="fdr_bh",
    run_cluster_tests=True,
    cluster_test_alternative="greater",
    cluster_fdr_scope="all",
    min_pairs_per_test=3,
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    ad = adata_T.copy()

    # --- optional cluster filtering ---
    if cluster_col is not None and exclude_clusters is not None:
        ad = ad[~ad.obs[cluster_col].isin(list(exclude_clusters))].copy()

    # --- compute MAPK score ---
    per_cell_score, genes_present, ad_scored = _score_genes_per_cell_scanpy(
        ad,
        genes=genes,
        score_key="MAPK_score",
        use_raw=use_raw,
        layer=layer,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
    )

    obs = ad_scored.obs[[specimen_col, "MAPK_score"]].copy()
    if cluster_col in ad_scored.obs:
        obs[cluster_col] = ad_scored.obs[cluster_col]

    obs["patient"] = patient_parser(obs[specimen_col].astype(str))
    obs["day"] = day_parser(obs[specimen_col].astype(str))

    if exclude_patients is not None:
        obs = obs[~obs["patient"].isin(list(exclude_patients))].copy()

    agg = "mean" if pseudobulk_agg == "mean" else "median"

    specimen_tbl = (
        obs.groupby(specimen_col)
        .agg(
            MAPK_score=("MAPK_score", agg),
            patient=("patient", "first"),
            day=("day", "first"),
            n_cells=("MAPK_score", "size"),
        )
        .reset_index()
    )

    wide_tbl = (
        specimen_tbl.pivot_table(index="patient", columns="day", values="MAPK_score", aggfunc="first")
        .rename(columns={d01: f"{d01}_MAPK", d15: f"{d15}_MAPK"})
        .reset_index()
    )

    if dropna_pairs:
        wide_tbl = wide_tbl.dropna(subset=[f"{d01}_MAPK", f"{d15}_MAPK"]).copy()

    wide_tbl["delta"] = wide_tbl[f"{d15}_MAPK"] - wide_tbl[f"{d01}_MAPK"]

    wide_tbl["group"] = "unknown"
    if nonresponder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(nonresponder_patients), "group"] = "nonresponder"
        wide_tbl.loc[~wide_tbl["patient"].isin(nonresponder_patients), "group"] = "responder"
    elif responder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(responder_patients), "group"] = "responder"
        wide_tbl.loc[~wide_tbl["patient"].isin(responder_patients), "group"] = "nonresponder"

    def _wilcox_delta(delta_series, label, alternative="two-sided"):
        d = delta_series.dropna().astype(float).values
        if len(d) < min_pairs_per_test:
            return {"test": label, "n": len(d), "stat": np.nan, "p": np.nan}
        if np.allclose(d, 0):
            return {"test": label, "n": len(d), "stat": 0.0, "p": 1.0}
        out = wilcoxon(d, zero_method="wilcox", alternative=alternative)
        return {"test": label, "n": len(d), "stat": float(out.statistic), "p": float(out.pvalue)}

    test_rows = []
    if run_overall:
        test_rows.append(_wilcox_delta(wide_tbl["delta"], "overall: D15-D01 MAPK vs 0"))

    if run_by_group:
        for g in ["responder", "nonresponder"]:
            test_rows.append(
                _wilcox_delta(
                    wide_tbl.loc[wide_tbl["group"] == g, "delta"],
                    f"{g}: D15-D01 MAPK vs 0"
                )
            )

    tests = pd.DataFrame(test_rows)

    if tests["p"].notna().any():
        tests["q"] = multipletests(tests["p"].fillna(1.0).values, method=fdr_method)[1]
    else:
        tests["q"] = np.nan

    return specimen_tbl, wide_tbl, tests


In [21]:
# Running functions detailed above on MAPK genes (9 gene score) 
# Nonresponders (PFS < 6 months) are below can interchange with Responders (PFS > 6 months)
MAPK_GENES = ["SPRY2", "SPRY4", "ETV4", "ETV5", "DUSP4", "DUSP6", "CCND1", "EPHA2", "EPHA4"]
nonresponders = ["P03","P14",'P15',"P19","P25","P27","P28","P29","P30","P33","P35","P36"]

MAPK_specimen_tbl, MAPK_wide_tbl, MAPK_test_tbl = mapk_change_stats_from_adata(
    adata_T=adata_CD8_T,
    genes=MAPK_GENES,
    cluster_col="sub_humap_fgraph_res.1",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients= None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
)
print(MAPK_meta)
print(MAPK_specimen_tbl)  # one row per specimenID (pseudobulked MAPK)
print(MAPK_wide_tbl)      # one row per patient with D01/D15 + delta
print(MAPK_test_tbl)             # Wilcoxon + BH q-values
MAPK_wide_tbl.to_csv('021726_CD8_MAPK_scores.csv', index=False)
print(len(adata_CD8_T))

{'genes_requested': ['SPRY2', 'SPRY4', 'ETV4', 'ETV5', 'DUSP4', 'DUSP6', 'CCND1', 'EPHA2', 'EPHA4'], 'genes_present': ['SPRY2', 'SPRY4', 'ETV4', 'ETV5', 'DUSP4', 'DUSP6', 'CCND1', 'EPHA2', 'EPHA4'], 'n_genes_present': 9, 'exclude_clusters': None, 'n_cells_used': 37421, 'pseudobulk_agg': 'median', 'use_raw': False, 'layer': None, 'score_method': 'scanpy.tl.score_genes', 'ctrl_size': 50, 'n_bins': 25, 'random_state': 0, 'min_cells_per_timepoint': 10, 'n_patients_kept': 21}
   specimenID  MAPK_score patient  day  n_cells
0      P02D01   -0.062809     P02  D01      294
1      P02D15   -0.061176     P02  D15      411
2      P03D01   -0.069415     P03  D01       43
3      P03D15   -0.057410     P03  D15      118
4      P07D01   -0.049505     P07  D01       36
5      P10D01   -0.065792     P10  D01       35
6      P10D15   -0.051713     P10  D15      152
7      P11D01   -0.045986     P11  D01      495
8      P11D15   -0.076653     P11  D15      461
9      P12D01   -0.041224     P12  D01      

In [24]:
# Running functions detailed above on pan-ISG score
# Nonresponders (PFS < 6 months) are below can interchange with Responders (PFS > 6 months)

ISG_GENES = ["HLA-DRB1","CD74","PSMB9","HLA-DQB1","HLA-DRA","HLA-B","PSMB10","HLA-DPA1",
    "B2M","HLA-F","PSME2","HLA-C","TAP2","TAP1","HLA-A","HLA-DRB5","HLA-DQA2",
    "CIITA","HLA-DQB2","HLA-DPB1","NOS2","GBP7","CCL8","GBP4","TRIM22","CASP1",
    "UBD","GBP1","IFNG","CCL23","XCL1","XCL2","STAT1","BST2","CX3CL1","TLR2",
    "GBP5","DAPK1","CCL3","CCL4","CCL20","IRF1","CCL5","SIRPA","IFI30","GBP2",
    "PARP9","IL12RB1","NR1H3","TRIM31","IRF9","IFITM3","GBP3","PARP14","CCL3L1",
    "EVL","SOCS1","TRIM21","IFITM1","STAT2","ZBP1","C19orf66","IFI35","IFIT2",
    "IFIT3","ISG15","IFNAR2","XAF1","IFI27","MX1","IFIT1","RSAD2","IFI6",
    "C10orf99","CXCL10","CXCL11","CXCL9","CXCL13","CXCL5","CXCL6"]

nonresponders = ["P03","P14",'P15',"P19","P25","P27","P28","P29","P30","P33","P35","P36"]

ISG_specimen_tbl, ISG_wide_tbl, ISG_test_tbl = mapk_change_stats_from_adata(
    adata_T=adata_CD8_T,
    genes=ISG_GENES,
    cluster_col="sub_humap_fgraph_res.1",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients= None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
)
print(ISG_specimen_tbl)  # one row per specimenID (pseudobulked ISG)
print(ISG_wide_tbl)      # one row per patient with D01/D15 + delta
print(ISG_test_tbl)             # Wilcoxon + BH q-values
ISG_wide_tbl.to_csv('021726_CD8_ISG_scores.csv', index=False)

   specimenID  MAPK_score patient  day  n_cells
0      P02D01    0.661032     P02  D01      294
1      P02D15    0.885413     P02  D15      411
2      P03D01    0.670636     P03  D01       43
3      P03D15    0.750382     P03  D15      118
4      P07D01    1.202699     P07  D01       36
5      P10D01    0.662518     P10  D01       35
6      P10D15    0.675670     P10  D15      152
7      P11D01    0.530470     P11  D01      495
8      P11D15    0.732694     P11  D15      461
9      P12D01    0.603301     P12  D01       73
10     P13D01    0.651141     P13  D01       59
11     P13D15    0.604875     P13  D15      107
12     P14D01    0.665857     P14  D01       25
13     P14D15    0.707845     P14  D15       43
14     P15D01    0.635309     P15  D01        9
15     P15D15    0.698110     P15  D15       11
16     P17D01    0.699633     P17  D01     1396
17     P17D15    0.995937     P17  D15      549
18     P18D01    0.604018     P18  D01      213
19     P18D15    0.878444     P18  D15  

In [25]:
# Compute gene signature score activity per cell on a per-cluster basis for the pan-ISG score.
# Question: are certain clusters responsible for significant shift in ISG score on treatment? If so, which ones?

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def _get_X_and_genes(adata, use_raw=False, layer=None):
    if use_raw and getattr(adata, "raw", None) is not None:
        X = adata.raw.X
        var_names = adata.raw.var_names
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names
    return X, var_names

def _score_genes_per_cell_scanpy(
    adata,
    genes,
    score_key="MAPK_score",
    use_raw=False,
    layer=None,
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Uses scanpy.sc.tl.score_genes to compute a per-cell gene-set score.
    Returns (scores_1d, genes_present, adata_scored).
    """
    import scanpy as sc

    ad = adata.copy()

    # If a layer is requested, score on that layer by temporarily swapping X.
    if layer is not None:
        if use_raw:
            raise ValueError("If use_raw=True, please do not pass layer (score_genes will use .raw).")
        if layer not in ad.layers:
            raise ValueError(f"layer='{layer}' not found in adata.layers")
        ad.X = ad.layers[layer]

    # Determine which gene names we are matching against
    if use_raw and getattr(ad, "raw", None) is not None:
        var_names = np.asarray(ad.raw.var_names)
        use_raw_arg = True
    else:
        var_names = np.asarray(ad.var_names)
        use_raw_arg = False

    gene_set = set(var_names.tolist())
    genes_present = [g for g in genes if g in gene_set]
    if len(genes_present) == 0:
        raise ValueError("None of the provided genes were found in adata.var_names (or adata.raw.var_names).")

    sc.tl.score_genes(
        ad,
        gene_list=genes_present,
        score_name=score_key,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
        use_raw=use_raw_arg,
        copy=False,
    )

    scores = ad.obs[score_key].to_numpy(dtype=float)
    return scores, genes_present, ad

def mapk_change_stats_from_adata(
    adata_mye,
    genes,
    specimen_col="specimenID",
    cluster_col="L2_humap_fgraph_res.0.4",
    exclude_clusters=None,  # filter OUT these clusters first
    patient_parser=lambda s: s.str.slice(0, 3),
    day_parser=lambda s: s.str.slice(3, 6),
    d01="D01",
    d15="D15",
    nonresponder_patients=None,
    responder_patients=None,
    exclude_patients=None,
    use_raw=False,
    layer=None,
    pseudobulk_agg="median",     # "mean" or "median" per specimen
    dropna_pairs=True,
    run_overall=True,
    run_by_group=True,
    fdr_method="fdr_bh",
    # per-cluster testing
    run_cluster_tests=True,
    cluster_test_alternative="greater",
    cluster_fdr_scope="all",
    min_pairs_per_test=3,
    # NEW: require >= this many cells in a patient×cluster×timepoint to include that pair
    min_cells_per_patient_cluster=3,
    # score_genes params
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Same as your function, but per-cluster tests enforce:
      For a given patient×cluster to be included, BOTH D01 and D15 must have
      >= min_cells_per_patient_cluster cells in that cluster (per timepoint).
    Returns:
      specimen_tbl, wide_tbl, tests, meta, cluster_tests, cluster_wide
    """

    ad = adata_mye.copy()

    # --- filter out clusters (optional) ---
    if cluster_col is not None and exclude_clusters is not None:
        if cluster_col not in ad.obs:
            raise ValueError(f"cluster_col='{cluster_col}' not found in adata.obs")
        ad = ad[~ad.obs[cluster_col].isin(list(exclude_clusters))].copy()

    # --- require specimen_col ---
    if specimen_col not in ad.obs:
        raise ValueError(f"specimen_col='{specimen_col}' not found in adata.obs")

    # --- compute per-cell score using scanpy ---
    per_cell_score, genes_present, ad_scored = _score_genes_per_cell_scanpy(
        ad,
        genes=genes,
        score_key="MAPK_score",
        use_raw=use_raw,
        layer=layer,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
    )

    # --- specimen-level pseudobulk (overall) ---
    obs_cols = [specimen_col, "MAPK_score"]
    if cluster_col is not None and cluster_col in ad_scored.obs.columns:
        obs_cols.append(cluster_col)

    obs = ad_scored.obs[obs_cols].copy()
    obs["patient"] = patient_parser(obs[specimen_col].astype(str))
    obs["day"] = day_parser(obs[specimen_col].astype(str))

    if exclude_patients is not None:
        obs = obs[~obs["patient"].isin(list(exclude_patients))].copy()

    agg = "mean" if pseudobulk_agg == "mean" else "median"

    specimen_tbl = (
        obs.groupby(specimen_col)
        .agg(
            MAPK_score=("MAPK_score", agg),
            patient=("patient", "first"),
            day=("day", "first"),
            n_cells=("MAPK_score", "size"),
        )
        .reset_index()
    )

    # --- patient x day wide table (overall) ---
    wide_tbl = (
        specimen_tbl.pivot_table(index="patient", columns="day", values="MAPK_score", aggfunc="first")
        .rename(columns={d01: f"{d01}_MAPK", d15: f"{d15}_MAPK"})
        .reset_index()
    )
    if dropna_pairs:
        wide_tbl = wide_tbl.dropna(subset=[f"{d01}_MAPK", f"{d15}_MAPK"]).copy()
    wide_tbl["delta"] = wide_tbl[f"{d15}_MAPK"] - wide_tbl[f"{d01}_MAPK"]

    # --- label responder status ---
    wide_tbl["group"] = "unknown"
    if nonresponder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(nonresponder_patients), "group"] = "nonresponder"
        wide_tbl.loc[~wide_tbl["patient"].isin(nonresponder_patients), "group"] = "responder"
    elif responder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(responder_patients), "group"] = "responder"
        wide_tbl.loc[~wide_tbl["patient"].isin(responder_patients), "group"] = "nonresponder"

    # --- paired Wilcoxon helper ---
    def _wilcox_delta(delta_series, label, alternative="two-sided"):
        d = delta_series.dropna().astype(float).values
        if len(d) < min_pairs_per_test:
            return {
                "test": label,
                "n": len(d),
                "stat": np.nan,
                "p": np.nan,
                "alternative": alternative,
                "median_delta": float(np.nanmedian(d)) if len(d) else np.nan,
            }
        if np.allclose(d, 0):
            return {
                "test": label,
                "n": len(d),
                "stat": 0.0,
                "p": 1.0,
                "alternative": alternative,
                "median_delta": float(np.median(d)),
            }
        out = wilcoxon(d, zero_method="wilcox", alternative=alternative)
        return {
            "test": label,
            "n": len(d),
            "stat": float(out.statistic),
            "p": float(out.pvalue),
            "alternative": alternative,
            "median_delta": float(np.median(d)),
        }

    # --- overall tests ---
    test_rows = []
    if run_overall:
        test_rows.append(_wilcox_delta(wide_tbl["delta"], "overall: D15-D01 MAPK vs 0", alternative="two-sided"))
    if run_by_group:
        for g in ["responder", "nonresponder"]:
            test_rows.append(
                _wilcox_delta(
                    wide_tbl.loc[wide_tbl["group"] == g, "delta"],
                    f"{g}: D15-D01 MAPK vs 0",
                    alternative="two-sided",
                )
            )
    tests = pd.DataFrame(test_rows)

    if tests["p"].notna().any():
        tests["q"] = multipletests(tests["p"].fillna(1.0).values, method=fdr_method)[1]
        tests["fdr_method"] = fdr_method
    else:
        tests["q"] = np.nan
        tests["fdr_method"] = fdr_method

    # =========================
    # Per-cluster testing WITH min cells per patient×cluster×timepoint
    # =========================
    cluster_tests = pd.DataFrame()
    cluster_wide = pd.DataFrame()

    if run_cluster_tests:
        if cluster_col is None or cluster_col not in obs.columns:
            raise ValueError("run_cluster_tests=True requires a valid cluster_col present in adata.obs.")

        # Pseudobulk per (specimen, cluster) + n_cells
        cl_spec = (
            obs.groupby([specimen_col, cluster_col])
            .agg(
                score=("MAPK_score", agg),
                patient=("patient", "first"),
                day=("day", "first"),
                n_cells=("MAPK_score", "size"),
            )
            .reset_index()
            .rename(columns={cluster_col: "cluster"})
        )

        # Pivot scores wide
        score_wide = (
            cl_spec.pivot_table(index=["patient", "cluster"], columns="day", values="score", aggfunc="first")
            .rename(columns={d01: f"{d01}_score", d15: f"{d15}_score"})
            .reset_index()
        )

        # Pivot n_cells wide
        ncell_wide = (
            cl_spec.pivot_table(index=["patient", "cluster"], columns="day", values="n_cells", aggfunc="first")
            .rename(columns={d01: f"{d01}_n_cells", d15: f"{d15}_n_cells"})
            .reset_index()
        )

        cluster_wide = score_wide.merge(ncell_wide, on=["patient", "cluster"], how="left")

        # label group
        cluster_wide["group"] = "unknown"
        if nonresponder_patients is not None:
            cluster_wide.loc[cluster_wide["patient"].isin(nonresponder_patients), "group"] = "nonresponder"
            cluster_wide.loc[~cluster_wide["patient"].isin(nonresponder_patients), "group"] = "responder"
        elif responder_patients is not None:
            cluster_wide.loc[cluster_wide["patient"].isin(responder_patients), "group"] = "responder"
            cluster_wide.loc[~cluster_wide["patient"].isin(responder_patients), "group"] = "nonresponder"

        # Require both scores + both n_cells, then enforce >= min cells at BOTH timepoints
        req = [f"{d01}_score", f"{d15}_score", f"{d01}_n_cells", f"{d15}_n_cells"]
        cluster_wide = cluster_wide.dropna(subset=req).copy()

        cluster_wide = cluster_wide[
            (cluster_wide[f"{d01}_n_cells"] >= min_cells_per_patient_cluster) &
            (cluster_wide[f"{d15}_n_cells"] >= min_cells_per_patient_cluster)
        ].copy()

        # deltas
        cluster_wide["delta"] = cluster_wide[f"{d15}_score"] - cluster_wide[f"{d01}_score"]

        # run wilcoxon per cluster overall and per group
        rows = []
        for cl in sorted(cluster_wide["cluster"].unique()):
            sub = cluster_wide.loc[cluster_wide["cluster"] == cl]

            rows.append(
                (_wilcox_delta(
                    sub["delta"],
                    label=f"cluster {cl} overall: D15-D01 score > 0",
                    alternative=cluster_test_alternative,
                ) | {"cluster": cl, "group": "overall"})
            )

            if run_by_group:
                for g in ["responder", "nonresponder"]:
                    rows.append(
                        (_wilcox_delta(
                            sub.loc[sub["group"] == g, "delta"],
                            label=f"cluster {cl} {g}: D15-D01 score > 0",
                            alternative=cluster_test_alternative,
                        ) | {"cluster": cl, "group": g})
                    )

        cluster_tests = pd.DataFrame(rows)

        # FDR on cluster tests
        if cluster_tests["p"].notna().any():
            if cluster_fdr_scope == "within_cluster":
                cluster_tests["q"] = np.nan
                for cl in cluster_tests["cluster"].unique():
                    mask = cluster_tests["cluster"] == cl
                    pvals = cluster_tests.loc[mask, "p"].fillna(1.0).values
                    cluster_tests.loc[mask, "q"] = multipletests(pvals, method=fdr_method)[1]
            else:
                pvals = cluster_tests["p"].fillna(1.0).values
                cluster_tests["q"] = multipletests(pvals, method=fdr_method)[1]

            cluster_tests["fdr_method"] = fdr_method
            cluster_tests["fdr_scope"] = cluster_fdr_scope
        else:
            cluster_tests["q"] = np.nan
            cluster_tests["fdr_method"] = fdr_method
            cluster_tests["fdr_scope"] = cluster_fdr_scope

        cluster_tests["significant_increase_q05"] = (cluster_tests["q"] < 0.05) & (cluster_tests["median_delta"] > 0)

        cluster_tests = (
            cluster_tests[
                ["cluster", "group", "n", "median_delta", "stat", "p", "q",
                 "significant_increase_q05", "alternative", "fdr_method", "fdr_scope"]
            ]
            .sort_values(["cluster", "group"])
            .reset_index(drop=True)
        )

    meta = {
        "genes_requested": list(genes),
        "genes_present": genes_present,
        "n_genes_present": len(genes_present),
        "exclude_clusters": list(exclude_clusters) if exclude_clusters is not None else None,
        "n_cells_used": int(ad_scored.n_obs),
        "pseudobulk_agg": pseudobulk_agg,
        "use_raw": use_raw,
        "layer": layer,
        "score_method": "scanpy.tl.score_genes",
        "ctrl_size": ctrl_size,
        "n_bins": n_bins,
        "random_state": random_state,
        "cluster_tests": bool(run_cluster_tests),
        "cluster_test_alternative": cluster_test_alternative,
        "cluster_fdr_scope": cluster_fdr_scope,
        "min_pairs_per_test": min_pairs_per_test,
        "min_cells_per_patient_cluster": min_cells_per_patient_cluster,
    }

    return specimen_tbl, wide_tbl, tests, meta, cluster_tests, cluster_wide


In [26]:
# Running functions detailed above on pan-ISG score on per-cluster basis, outputs saved
ISG_GENES = ["HLA-DRB1","CD74","PSMB9","HLA-DQB1","HLA-DRA","HLA-B","PSMB10","HLA-DPA1",
             "B2M","HLA-F","PSME2","HLA-C","TAP2","TAP1","HLA-A","HLA-DRB5","HLA-DQA2",
             "CIITA","HLA-DQB2","HLA-DPB1","NOS2","GBP7","CCL8","GBP4","TRIM22","CASP1",
             "UBD","GBP1","IFNG","CCL23","XCL1","XCL2","STAT1","BST2","CX3CL1","TLR2",
             "GBP5","DAPK1","CCL3","CCL4","CCL20","IRF1","CCL5","SIRPA","IFI30","GBP2",
             "PARP9","IL12RB1","NR1H3","TRIM31","IRF9","IFITM3","GBP3","PARP14","CCL3L1",
             "EVL","SOCS1","TRIM21","IFITM1","STAT2","ZBP1","C19orf66","IFI35","IFIT2",
             "IFIT3","ISG15","IFNAR2","XAF1","IFI27","MX1","IFIT1","RSAD2","IFI6",
             "C10orf99","CXCL10","CXCL11","CXCL9","CXCL13","CXCL5","CXCL6"]

nonresponders = ["P03","P14","P15","P19","P25","P27","P28","P29","P30","P33","P35","P36"]

ISG_specimen_tbl, ISG_wide_tbl, ISG_test_tbl, ISG_meta, ISG_cluster_tests, ISG_cluster_wide = mapk_change_stats_from_adata(
    adata_mye=adata_CD8_T,
    genes=ISG_GENES,
    cluster_col="sub_humap_fgraph_res.1",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients=None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
    run_cluster_tests=True,
    cluster_test_alternative="greater",   # <-- tests for increase
    cluster_fdr_scope="all",              # BH across all cluster+group tests
)

print(ISG_meta)

# Existing outputs
print(ISG_specimen_tbl.head())
print(ISG_wide_tbl.head())
print(ISG_test_tbl)

# NEW: this is the table you asked for (responder/nonresponder per cluster)
print(ISG_cluster_tests)

# Save what you want
ISG_cluster_tests.to_csv("021726_CD8_ISG_cluster_increase_tests.csv", index=False)


{'genes_requested': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4', 'TRIM22', 'CASP1', 'UBD', 'GBP1', 'IFNG', 'CCL23', 'XCL1', 'XCL2', 'STAT1', 'BST2', 'CX3CL1', 'TLR2', 'GBP5', 'DAPK1', 'CCL3', 'CCL4', 'CCL20', 'IRF1', 'CCL5', 'SIRPA', 'IFI30', 'GBP2', 'PARP9', 'IL12RB1', 'NR1H3', 'TRIM31', 'IRF9', 'IFITM3', 'GBP3', 'PARP14', 'CCL3L1', 'EVL', 'SOCS1', 'TRIM21', 'IFITM1', 'STAT2', 'ZBP1', 'C19orf66', 'IFI35', 'IFIT2', 'IFIT3', 'ISG15', 'IFNAR2', 'XAF1', 'IFI27', 'MX1', 'IFIT1', 'RSAD2', 'IFI6', 'C10orf99', 'CXCL10', 'CXCL11', 'CXCL9', 'CXCL13', 'CXCL5', 'CXCL6'], 'genes_present': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4

# CD4 T cell gene signature scores Figure 3H (part of 3J), Figure S3D, S3F

In [27]:
cd4_coords = pd.read_csv('gs://fc-e9295d8f-5730-4967-b1ce-22c3d775d7c2/HC_Tclus/cd4_t_full.csv', sep=',')

In [29]:
# pruning doublets in clusters 7 and 11, renumbering
col = "humap_fgraph_res.0.9"  # change if different

# 1) Remove clusters 7 and 11
cd4_coords_drop_doublets = cd4_coords[
    ~cd4_coords[col].isin([7, 11])
].copy()

x = cd4_coords_drop_doublets[col].astype(int)

# 2) Shift 8,9,10 down by 1
cd4_coords_drop_doublets[col] = np.select(
    [
        x.isin([8, 9, 10]),
    ],
    [
        x - 1,
    ],
    default=x
).astype(int)

# Sanity check
print("Unique clusters:", sorted(cd4_coords_drop_doublets[col].unique()))
print("Min:", cd4_coords_drop_doublets[col].min())
print("Max:", cd4_coords_drop_doublets[col].max())
print("Total clusters:", len(cd4_coords_drop_doublets[col].unique()))


Unique clusters: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Min: 0
Max: 9
Total clusters: 10


In [30]:
# merging cd4 dataframe with adata that has been filtered to just include all T cells 

cluster_col = "humap_fgraph_res.0.9"

def merge_cell_df_into_adata_obs(
    adata,
    cell_df,
    cols=None,
    key_df=None,          # e.g. "barcode" / "cell_id" if df doesn't use index as cell id
    key_adata=None,       # e.g. "barcode" / "cell_id" if adata.obs has a column for it
    overwrite=False,
    verbose=True,):
    """
    Merge columns from a cell-level DataFrame into adata.obs.

    Best-case:
      - cell_df.index == adata.obs_names  (unique)
    Otherwise:
      - specify key_df (column in cell_df) and key_adata (column in adata.obs)
        that contain the same cell IDs.

    Returns: adata (modified in place) + a small meta dict
    """
    df = cell_df.copy()

    # choose columns to merge
    if cols is None:
        cols = list(df.columns)
    else:
        cols = [c for c in cols if c in df.columns]
        if len(cols) == 0:
            raise ValueError("None of the requested `cols` exist in cell_df.")

    # Case 1: merge by index (cell_df.index -> adata.obs_names)
    can_index_join = df.index.is_unique and pd.Index(df.index).isin(adata.obs_names).mean() > 0.50
    if key_df is None and key_adata is None and can_index_join:
        df_sub = df[cols].copy()
        # align exactly to adata.obs_names
        df_sub = df_sub.reindex(adata.obs_names)

        # avoid clobbering
        for c in cols:
            if (c in adata.obs.columns) and (not overwrite):
                if verbose:
                    print(f"[merge] skipping existing column (overwrite=False): {c}")
                df_sub = df_sub.drop(columns=[c])

        adata.obs = adata.obs.join(df_sub, how="left")

        meta = {
            "mode": "index_join",
            "n_cells_adata": int(adata.n_obs),
            "n_cells_df": int(df.shape[0]),
            "frac_df_index_in_adata": float(pd.Index(df.index).isin(adata.obs_names).mean()),
            "n_cols_added": int(df_sub.shape[1]),
        }
        if verbose:
            print("[merge] merged by index into adata.obs")
            print(meta)
        return adata, meta

    # Case 2: merge by explicit keys
    if key_df is None or key_adata is None:
        raise ValueError(
            "Could not confidently merge by index. "
            "Provide `key_df` (column in cell_df) and `key_adata` (column in adata.obs) "
            "that both contain the same cell IDs."
        )

    if key_adata not in adata.obs.columns:
        raise ValueError(f"key_adata='{key_adata}' not found in adata.obs columns.")
    if key_df not in df.columns:
        raise ValueError(f"key_df='{key_df}' not found in cell_df columns.")

    df_sub = df[[key_df] + cols].copy()
    df_sub = df_sub.drop_duplicates(subset=[key_df])

    # avoid clobbering
    cols_to_add = []
    for c in cols:
        if (c in adata.obs.columns) and (not overwrite):
            if verbose:
                print(f"[merge] skipping existing column (overwrite=False): {c}")
        else:
            cols_to_add.append(c)

    df_sub = df_sub[[key_df] + cols_to_add]

    merged = adata.obs.reset_index().rename(columns={"index": "__obsname__"})
    merged = merged.merge(df_sub, how="left", left_on=key_adata, right_on=key_df)
    merged = merged.set_index("__obsname__")
    merged = merged.drop(columns=[key_df])

    adata.obs = merged

    frac_matched = float(adata.obs[cols_to_add].notna().mean().mean()) if cols_to_add else 1.0
    meta = {
        "mode": "key_merge",
        "key_adata": key_adata,
        "key_df": key_df,
        "n_cells_adata": int(adata.n_obs),
        "n_cells_df": int(df.shape[0]),
        "n_cols_added": int(len(cols_to_add)),
        "avg_nonnull_rate_added_cols": frac_matched,
    }
    if verbose:
        print("[merge] merged by keys into adata.obs")
        print(meta)
    return adata, meta

adata_T = sc.read_h5ad("adata_T.h5ad")
adata_CD4_T, merge_meta = merge_cell_df_into_adata_obs(
    adata=adata_T,
    cell_df=cd4_coords_drop_doublets,
    cols=[cluster_col],
    key_df="cell_id",        # <-- change
    key_adata="Unnamed: 0",     # <-- change
    overwrite=True,
    verbose=True,
)

# subset adata to only the cells present in cd4_coords_drop_doublets
cells_keep = pd.Index(cd4_coords_drop_doublets["cell_id"].astype(str).unique())

# if your adata obs key isn't string, normalize it
adata_keys = adata_CD4_T.obs["Unnamed: 0"].astype(str)

mask = adata_keys.isin(cells_keep).values
adata_CD4_T = adata_CD4_T[mask].copy()

print("After subsetting:", adata_CD4_T.n_obs, "cells")
print("Expected (unique df cells):", len(cells_keep))



[merge] merged by keys into adata.obs
{'mode': 'key_merge', 'key_adata': 'Unnamed: 0', 'key_df': 'cell_id', 'n_cells_adata': 134270, 'n_cells_df': 57074, 'n_cols_added': 1, 'avg_nonnull_rate_added_cols': 0.4250688910404409}
After subsetting: 57074 cells
Expected (unique df cells): 57074


In [31]:
# Compute gene signature score activity per cell using Scanpy gene-set scoring, then pseudobulk scores to the specimen 
# level (mean/median across cells). Paired D01 vs D15 changes are calculated per patient, and significance is 
# tested using a paired Wilcoxon test overall and by PFS > 6 months or < 6 months group with BH-FDR correction. 

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def _get_X_and_genes(adata, use_raw=False, layer=None):
    if use_raw and getattr(adata, "raw", None) is not None:
        X = adata.raw.X
        var_names = adata.raw.var_names
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names
    return X, var_names


def _score_genes_per_cell_scanpy(
    adata,
    genes,
    score_key="MAPK_score",
    use_raw=False,
    layer=None,
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Uses scanpy.sc.tl.score_genes to compute a per-cell gene-set score.
    Returns (scores_1d, genes_present, adata_scored).
    """
    ad = adata.copy()

    # If a layer is requested, score on that layer by temporarily swapping X.
    # (scanpy's score_genes historically scores on .X; use_raw overrides.)
    if layer is not None:
        if use_raw:
            raise ValueError("If use_raw=True, please do not pass layer (score_genes will use .raw).")
        if layer not in ad.layers:
            raise ValueError(f"layer='{layer}' not found in adata.layers")
        ad.X = ad.layers[layer]

    # Determine which gene names we are matching against
    if use_raw and getattr(ad, "raw", None) is not None:
        var_names = np.asarray(ad.raw.var_names)
        use_raw_arg = True
    else:
        var_names = np.asarray(ad.var_names)
        use_raw_arg = False

    gene_set = set(var_names.tolist())
    genes_present = [g for g in genes if g in gene_set]
    if len(genes_present) == 0:
        raise ValueError("None of the provided genes were found in adata.var_names (or adata.raw.var_names).")

    # Score genes
    sc.tl.score_genes(
        ad,
        gene_list=genes_present,
        score_name=score_key,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
        use_raw=use_raw_arg,
        copy=False,
    )

    scores = ad.obs[score_key].to_numpy(dtype=float)
    return scores, genes_present, ad

def mapk_change_stats_from_adata(
    adata_T,
    genes,
    specimen_col="specimenID",
    cluster_col="sub_humap_fgraph_res.1",
    exclude_clusters=None,                 # filter OUT these clusters first
    patient_parser=lambda s: s.str.slice(0, 3),
    day_parser=lambda s: s.str.slice(3, 6),
    d01="D01",
    d15="D15",
    nonresponder_patients=None,
    responder_patients=None,
    exclude_patients=None,                 # e.g. ["P15","P23"]
    use_raw=False,
    layer=None,
    pseudobulk_agg="median",               # "mean" or "median" per specimen
    dropna_pairs=True,
    run_overall=True,
    run_by_group=True,
    fdr_method="fdr_bh",
    # NEW: min cells requirement per patient per timepoint
    min_cells_per_timepoint=10,            # require >=10 cells at D01 AND >=10 cells at D15 for patient to be included
    # score_genes params
    ctrl_size=50,
    n_bins=25,
    random_state=0,):
    """
    Computes MAPK score per cell via sc.tl.score_genes, then pseudobulks to 1 value per specimen,
    then tests paired change (D15 - D01) per patient using Wilcoxon, with BH-FDR.

    NEW: filters to patients that have >= min_cells_per_timepoint cells at BOTH D01 and D15.
    """

    ad = adata_T.copy()

    # --- filter out clusters (optional) ---
    if cluster_col is not None and exclude_clusters is not None:
        if cluster_col not in ad.obs:
            raise ValueError(f"cluster_col='{cluster_col}' not found in adata.obs")
        ad = ad[~ad.obs[cluster_col].isin(list(exclude_clusters))].copy()

    # --- require specimen_col ---
    if specimen_col not in ad.obs:
        raise ValueError(f"specimen_col='{specimen_col}' not found in adata.obs")

    # --- compute per-cell MAPK score using scanpy ---
    per_cell_score, genes_present, ad_scored = _score_genes_per_cell_scanpy(
        ad,
        genes=genes,
        score_key="MAPK_score",
        use_raw=use_raw,
        layer=layer,
        ctrl_size=ctrl_size,
        n_bins=n_bins,
        random_state=random_state,
    )

    # --- specimen-level table (cell-level info needed for counting) ---
    obs = ad_scored.obs[[specimen_col, "MAPK_score"]].copy()
    obs["patient"] = patient_parser(obs[specimen_col].astype(str))
    obs["day"] = day_parser(obs[specimen_col].astype(str))

    if exclude_patients is not None:
        obs = obs[~obs["patient"].isin(list(exclude_patients))].copy()

    # --- NEW: require >= min_cells_per_timepoint cells at D01 and D15 per patient ---
    cell_counts = (
        obs.groupby(["patient", "day"])
           .size()
           .unstack(fill_value=0)
    )

    # some patients may lack D01/D15 columns entirely; guard with .get
    d01_counts = cell_counts[d01] if d01 in cell_counts.columns else pd.Series(0, index=cell_counts.index)
    d15_counts = cell_counts[d15] if d15 in cell_counts.columns else pd.Series(0, index=cell_counts.index)

    keep_patients = cell_counts.index[
        (d01_counts >= min_cells_per_timepoint) & (d15_counts >= min_cells_per_timepoint)
    ].tolist()

    obs = obs[obs["patient"].isin(keep_patients)].copy()

    # --- specimen-level pseudobulk ---
    if pseudobulk_agg == "mean":
        specimen_tbl = (
            obs.groupby(specimen_col)
               .agg(
                    MAPK_score=("MAPK_score", "mean"),
                    patient=("patient", "first"),
                    day=("day", "first"),
                    n_cells=("MAPK_score", "size"),
               )
               .reset_index()
        )
    elif pseudobulk_agg == "median":
        specimen_tbl = (
            obs.groupby(specimen_col)
               .agg(
                    MAPK_score=("MAPK_score", "median"),
                    patient=("patient", "first"),
                    day=("day", "first"),
                    n_cells=("MAPK_score", "size"),
               )
               .reset_index()
        )
    else:
        raise ValueError("pseudobulk_agg must be 'mean' or 'median'")

    # --- patient x day wide table ---
    wide_tbl = (
        specimen_tbl.pivot_table(index="patient", columns="day", values="MAPK_score", aggfunc="first")
                    .rename(columns={d01: f"{d01}_MAPK", d15: f"{d15}_MAPK"})
                    .reset_index()
    )

    if dropna_pairs:
        wide_tbl = wide_tbl.dropna(subset=[f"{d01}_MAPK", f"{d15}_MAPK"]).copy()

    wide_tbl["delta"] = wide_tbl[f"{d15}_MAPK"] - wide_tbl[f"{d01}_MAPK"]

    # --- label responder status ---
    wide_tbl["group"] = "unknown"
    if nonresponder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(nonresponder_patients), "group"] = "nonresponder"
        wide_tbl.loc[~wide_tbl["patient"].isin(nonresponder_patients), "group"] = "responder"
    elif responder_patients is not None:
        wide_tbl.loc[wide_tbl["patient"].isin(responder_patients), "group"] = "responder"
        wide_tbl.loc[~wide_tbl["patient"].isin(responder_patients), "group"] = "nonresponder"

    # --- paired Wilcoxon on delta vs 0 ---
    def _wilcox_delta(delta_series, label):
        d = delta_series.dropna().astype(float).values
        if len(d) < 3:
            return {"test": label, "n": len(d), "stat": np.nan, "p": np.nan}
        if np.allclose(d, 0):
            return {"test": label, "n": len(d), "stat": 0.0, "p": 1.0}
        try:
            out = wilcoxon(d, zero_method="wilcox", alternative="two-sided")
            return {"test": label, "n": len(d), "stat": float(out.statistic), "p": float(out.pvalue)}
        except Exception:
            return {"test": label, "n": len(d), "stat": np.nan, "p": np.nan}

    test_rows = []
    if run_overall:
        test_rows.append(_wilcox_delta(wide_tbl["delta"], "overall: D15-D01 MAPK vs 0"))
    if run_by_group:
        for g in ["responder", "nonresponder"]:
            test_rows.append(
                _wilcox_delta(
                    wide_tbl.loc[wide_tbl["group"] == g, "delta"],
                    f"{g}: D15-D01 MAPK vs 0"
                )
            )

    tests = pd.DataFrame(test_rows)

    # --- BH-FDR across the tests we ran ---
    if tests["p"].notna().any():
        pvals = tests["p"].fillna(1.0).values
        tests["q"] = multipletests(pvals, method=fdr_method)[1]
        tests["fdr_method"] = fdr_method
    else:
        tests["q"] = np.nan
        tests["fdr_method"] = fdr_method

    meta = {
        "genes_requested": list(genes),
        "genes_present": genes_present,
        "n_genes_present": len(genes_present),
        "exclude_clusters": list(exclude_clusters) if exclude_clusters is not None else None,
        "n_cells_used": int(obs.shape[0]),
        "pseudobulk_agg": pseudobulk_agg,
        "use_raw": use_raw,
        "layer": layer,
        "score_method": "scanpy.tl.score_genes",
        "ctrl_size": ctrl_size,
        "n_bins": n_bins,
        "random_state": random_state,
        "min_cells_per_timepoint": min_cells_per_timepoint,
        "n_patients_kept": len(keep_patients),
    }

    return specimen_tbl, wide_tbl, tests, meta


In [32]:
# Running functions detailed above on MAPK genes (9 gene score) 
# Nonresponders (PFS < 6 months) are below can interchange with Responders (PFS > 6 months)

MAPK_GENES = ["SPRY2", "SPRY4", "ETV4", "ETV5", "DUSP4", "DUSP6", "CCND1", "EPHA2", "EPHA4"]
nonresponders = ["P03","P14",'P15',"P19","P25","P27","P28","P29","P30","P33","P35","P36"]

MAPK_specimen_tbl, MAPK_wide_tbl, MAPK_test_tbl, MAPK_meta = mapk_change_stats_from_adata(
    adata_T=adata_CD4_T,
    genes=MAPK_GENES,
    cluster_col="humap_fgraph_res.0.9",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients= None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
)
print(MAPK_meta)
print(MAPK_specimen_tbl)  # one row per specimenID (pseudobulked MAPK)
print(MAPK_wide_tbl)      # one row per patient with D01/D15 + delta
print(MAPK_test_tbl)             # Wilcoxon + BH q-values
MAPK_wide_tbl.to_csv('021726_CD4_MAPK_scores.csv', index=False)

{'genes_requested': ['SPRY2', 'SPRY4', 'ETV4', 'ETV5', 'DUSP4', 'DUSP6', 'CCND1', 'EPHA2', 'EPHA4'], 'genes_present': ['SPRY2', 'SPRY4', 'ETV4', 'ETV5', 'DUSP4', 'DUSP6', 'CCND1', 'EPHA2', 'EPHA4'], 'n_genes_present': 9, 'exclude_clusters': None, 'n_cells_used': 51023, 'pseudobulk_agg': 'median', 'use_raw': False, 'layer': None, 'score_method': 'scanpy.tl.score_genes', 'ctrl_size': 50, 'n_bins': 25, 'random_state': 0, 'min_cells_per_timepoint': 10, 'n_patients_kept': 22}
   specimenID  MAPK_score patient   day  n_cells
0      P02D01   -0.066288     P02   D01      654
1      P02D15   -0.071633     P02   D15      589
2      P03D01   -0.068641     P03   D01       73
3      P03D15   -0.084994     P03   D15      169
4      P07D01         NaN    None  None        0
5      P10D01    0.101513     P10   D01       30
6      P10D15   -0.068219     P10   D15      291
7      P11D01   -0.053272     P11   D01     2105
8      P11D15   -0.085351     P11   D15     2264
9      P12D01         NaN    None 

In [33]:
# Running functions detailed above on pan-ISG score
# Nonresponders (PFS < 6 months) are below can interchange with Responders (PFS > 6 months)

ISG_GENES = ["HLA-DRB1","CD74","PSMB9","HLA-DQB1","HLA-DRA","HLA-B","PSMB10","HLA-DPA1",
    "B2M","HLA-F","PSME2","HLA-C","TAP2","TAP1","HLA-A","HLA-DRB5","HLA-DQA2",
    "CIITA","HLA-DQB2","HLA-DPB1","NOS2","GBP7","CCL8","GBP4","TRIM22","CASP1",
    "UBD","GBP1","IFNG","CCL23","XCL1","XCL2","STAT1","BST2","CX3CL1","TLR2",
    "GBP5","DAPK1","CCL3","CCL4","CCL20","IRF1","CCL5","SIRPA","IFI30","GBP2",
    "PARP9","IL12RB1","NR1H3","TRIM31","IRF9","IFITM3","GBP3","PARP14","CCL3L1",
    "EVL","SOCS1","TRIM21","IFITM1","STAT2","ZBP1","C19orf66","IFI35","IFIT2",
    "IFIT3","ISG15","IFNAR2","XAF1","IFI27","MX1","IFIT1","RSAD2","IFI6",
    "C10orf99","CXCL10","CXCL11","CXCL9","CXCL13","CXCL5","CXCL6"]

nonresponders = ["P03","P14",'P15',"P19","P25","P27","P28","P29","P30","P33","P35","P36"]

ISG_specimen_tbl, ISG_wide_tbl, ISG_test_tbl, ISG_meta = mapk_change_stats_from_adata(
    adata_T=adata_CD4_T,
    genes=ISG_GENES,
    cluster_col="humap_fgraph_res.0.9",
    exclude_clusters=None,
    nonresponder_patients=nonresponders,
    exclude_patients= None,
    pseudobulk_agg="median",
    fdr_method="fdr_bh",
    use_raw=False,
    layer=None,
)
print(ISG_meta)
print(ISG_specimen_tbl)  # one row per specimenID (pseudobulked ISG)
print(ISG_wide_tbl)      # one row per patient with D01/D15 + delta
print(ISG_test_tbl)             # Wilcoxon + BH q-values
ISG_wide_tbl.to_csv('021726_CD4_ISG_scores.csv', index=False)

{'genes_requested': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4', 'TRIM22', 'CASP1', 'UBD', 'GBP1', 'IFNG', 'CCL23', 'XCL1', 'XCL2', 'STAT1', 'BST2', 'CX3CL1', 'TLR2', 'GBP5', 'DAPK1', 'CCL3', 'CCL4', 'CCL20', 'IRF1', 'CCL5', 'SIRPA', 'IFI30', 'GBP2', 'PARP9', 'IL12RB1', 'NR1H3', 'TRIM31', 'IRF9', 'IFITM3', 'GBP3', 'PARP14', 'CCL3L1', 'EVL', 'SOCS1', 'TRIM21', 'IFITM1', 'STAT2', 'ZBP1', 'C19orf66', 'IFI35', 'IFIT2', 'IFIT3', 'ISG15', 'IFNAR2', 'XAF1', 'IFI27', 'MX1', 'IFIT1', 'RSAD2', 'IFI6', 'C10orf99', 'CXCL10', 'CXCL11', 'CXCL9', 'CXCL13', 'CXCL5', 'CXCL6'], 'genes_present': ['HLA-DRB1', 'CD74', 'PSMB9', 'HLA-DQB1', 'HLA-DRA', 'HLA-B', 'PSMB10', 'HLA-DPA1', 'B2M', 'HLA-F', 'PSME2', 'HLA-C', 'TAP2', 'TAP1', 'HLA-A', 'HLA-DRB5', 'HLA-DQA2', 'CIITA', 'HLA-DQB2', 'HLA-DPB1', 'NOS2', 'GBP7', 'CCL8', 'GBP4